# AvitoTech ML CUP 2026 — Safe Neural Solution on a Real Data Slice

This notebook trains one neural recommender on a controlled slice of the official data and writes `submission.csv`.

Memory strategy:
1. Stream a small, real slice of train events + all eval events.
2. Build a bounded vocabulary of the most useful items.
3. Read `item_features.parquet` **in batches** and keep features only for vocab items.
4. Train one end-to-end neural network with item metadata + user event sequence.
5. Neural-rerank a bounded candidate pool and save a full submission.

In [ ]:
# =========================
# 0. CONFIG: edit here only
# =========================
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/datasets/nikitakuznetsof/avito-ml-cup-2026')
SUBMISSION_FILE = 'submission.csv'
MODEL_DIR = Path('model_artifacts'); MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR = Path('reports'); REPORT_DIR.mkdir(exist_ok=True)

SEED = 42

# Real-data slice. Designed to run first time without RAM crashes on 2xT4 Kaggle.
TRAIN_PARTITION_LIMIT = 12
PARQUET_BATCH_ROWS = 120_000
MAX_VOCAB_ITEMS = 350_000
MAX_TRAIN_EXAMPLES = 500_000
MAX_SEQ_LEN = 40

# Neural net.
BATCH_SIZE = 512
EPOCHS = 4
NEGATIVES = 32
EMB_DIM = 96
HIDDEN_DIM = 160
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

# Prediction.
SUBMISSION_K = 160
CANDIDATE_POOL_SIZE = 8192
GLOBAL_CANDIDATES = 1200
PER_FEATURE_CANDIDATES = 250

# Smart item metadata features. These are official item_features columns.
ITEM_FEATURE_COLS = [
    'vertical_id', 'category_ext_y', 'region_id_y', 'loc_id_y',
    'sid_0_y', 'sid_1_y', 'sid_2_y', 'sid_3_y',
]
VALID_VERTICALS = {0, 2, 3, 4, 5, 7}

print('DATA_ROOT:', DATA_ROOT)
print('SUBMISSION_FILE:', SUBMISSION_FILE)


In [ ]:
# =========================
# 1. Imports and environment
# =========================
import gc, json, random
from collections import Counter, defaultdict, deque
from typing import Dict, List

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE, 'cuda_count:', torch.cuda.device_count())
if DEVICE.type == 'cuda':
    for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))


In [ ]:
# =========================
# 2. Paths
# =========================
def must_exist(path: Path) -> Path:
    if not path.exists(): raise FileNotFoundError(path)
    return path

def find_first(patterns):
    for pattern in patterns:
        hits = sorted(DATA_ROOT.glob(pattern))
        if hits: return hits[0]
    raise FileNotFoundError(patterns)

EVENT_COLUMNS = ['timestamp', 'eid', 'user_id', 'item_id']
ITEM_ID_COL = 'item_id'
ITEMS_PATH = must_exist(DATA_ROOT / 'item_features.parquet')
EVAL_USERS_PATH = must_exist(DATA_ROOT / 'eval_users.csv')
CONTACT_EIDS_PATH = must_exist(DATA_ROOT / 'contact_eids.csv')
EVAL_EVENTS_PATH = find_first(['eval_user_events/eval_user_events.pq', 'eval_user_events.pq', 'eval_user_events.parquet'])
all_train_paths = sorted(DATA_ROOT.glob('train_*/*/part_*.parquet')) or sorted(DATA_ROOT.glob('train_data/part_*.parquet'))
# Fresh train partitions are much more useful for Recall@160 than old part_000... files.
TRAIN_PATHS = list(reversed(all_train_paths))[:TRAIN_PARTITION_LIMIT]

print('ITEMS_PATH:', ITEMS_PATH)
print('EVAL_EVENTS_PATH:', EVAL_EVENTS_PATH)
print('train partitions selected:', len(TRAIN_PATHS))


In [ ]:
# =========================
# 3. Eval users and target eids
# =========================
eval_users_df = pd.read_csv(EVAL_USERS_PATH)
USER_COL = 'user_id' if 'user_id' in eval_users_df.columns else eval_users_df.columns[0]
eval_users = eval_users_df[USER_COL].astype('int64').to_numpy()
eval_user_set = set(map(int, eval_users))

contact_df = pd.read_csv(CONTACT_EIDS_PATH)
CONTACT_COL = 'mapped_eid' if 'mapped_eid' in contact_df.columns else contact_df.columns[0]
contact_eids = set(contact_df[CONTACT_COL].astype(int).tolist())
print('eval users:', len(eval_users))
print('contact eids:', sorted(contact_eids))


In [ ]:
# =========================
# 4. Stream events: bounded item vocabulary + training examples + eval histories
# =========================
item_to_idx: Dict[int, int] = {}
idx_to_item = [-1]
item_counts = Counter(); contact_item_counts = Counter(); eid_counts = Counter()

def get_item_idx(item_id: int) -> int:
    item_id = int(item_id)
    idx = item_to_idx.get(item_id)
    if idx is not None: return idx
    if len(idx_to_item) <= MAX_VOCAB_ITEMS:
        idx = len(idx_to_item); item_to_idx[item_id] = idx; idx_to_item.append(item_id); return idx
    return 0

class ExampleStore:
    def __init__(self, max_examples, seq_len):
        self.x_items = np.zeros((max_examples, seq_len), dtype=np.int32)
        self.x_eids = np.zeros((max_examples, seq_len), dtype=np.int16)
        self.y = np.zeros(max_examples, dtype=np.int32)
        self.n = 0; self.max_examples = max_examples; self.seq_len = seq_len
    def add(self, hist, pos):
        if self.n >= self.max_examples or pos <= 0 or not hist: return
        h = list(hist)[-self.seq_len:]; off = self.seq_len - len(h)
        self.x_items[self.n, off:] = [x[0] for x in h]
        self.x_eids[self.n, off:] = [x[1] for x in h]
        self.y[self.n] = pos; self.n += 1
    def finish(self): return self.x_items[:self.n], self.x_eids[:self.n], self.y[:self.n]

examples = ExampleStore(MAX_TRAIN_EXAMPLES, MAX_SEQ_LEN)
eval_histories: Dict[int, deque] = {int(u): deque(maxlen=MAX_SEQ_LEN) for u in eval_users}
print('preallocated train arrays MB:', round((examples.x_items.nbytes + examples.x_eids.nbytes + examples.y.nbytes)/1024**2, 1))

def iter_event_batches(paths):
    for path in paths:
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=PARQUET_BATCH_ROWS, columns=EVENT_COLUMNS):
            df = batch.to_pandas()
            yield path, df
            del df; gc.collect()

def process_events(df: pd.DataFrame, keep_eval_histories: bool, only_eval_users: bool) -> int:
    if only_eval_users:
        df = df[df['user_id'].isin(eval_user_set)]
    if len(df) == 0: return 0
    df = df.sort_values(['user_id', 'timestamp'])
    used = 0; last_user = None; hist = deque(maxlen=MAX_SEQ_LEN)
    for user_id, eid, item_id in df[['user_id','eid','item_id']].itertuples(index=False, name=None):
        user_id = int(user_id); eid = int(eid)
        if user_id != last_user:
            hist = eval_histories[user_id] if keep_eval_histories and user_id in eval_histories else deque(maxlen=MAX_SEQ_LEN)
            last_user = user_id
        idx = get_item_idx(int(item_id))
        if idx <= 0: continue
        item_counts[idx] += 1; eid_counts[eid] += 1
        if eid in contact_eids:
            contact_item_counts[idx] += 1; examples.add(hist, idx)
        hist.append((idx, min(eid, 32767))); used += 1
    return used

train_used = 0
for path, df in tqdm(iter_event_batches(TRAIN_PATHS), desc='train batches'):
    if examples.n >= MAX_TRAIN_EXAMPLES:
        del df; break
    train_used += process_events(df, keep_eval_histories=False, only_eval_users=False)
    del df; gc.collect()

eval_used = 0
for path, df in tqdm(iter_event_batches([EVAL_EVENTS_PATH]), desc='eval batches'):
    eval_used += process_events(df, keep_eval_histories=True, only_eval_users=True)
    del df; gc.collect()

X_items, X_eids, y = examples.finish()
idx_to_item = np.asarray(idx_to_item, dtype=np.int64)
print('vocab items:', len(idx_to_item)-1)
print('train_used:', train_used, 'eval_used:', eval_used, 'examples:', len(y))
print('top eids:', eid_counts.most_common(10))
if len(y) < 5000: raise RuntimeError('Too few examples; increase TRAIN_PARTITION_LIMIT')
del examples; gc.collect()


In [ ]:
# =========================
# 5. Read item_features only for vocabulary items, in parquet batches
# =========================
schema_names = pq.ParquetFile(ITEMS_PATH).schema.names
feature_cols = [c for c in ITEM_FEATURE_COLS if c in schema_names]
print('using item feature columns:', feature_cols)

vocab_item_set = set(map(int, idx_to_item[1:]))
item_feature_rows = []
pf = pq.ParquetFile(ITEMS_PATH)
read_cols = [ITEM_ID_COL] + feature_cols
for batch in tqdm(pf.iter_batches(batch_size=120_000, columns=read_cols), desc='item feature batches'):
    part = batch.to_pandas()
    part = part[part[ITEM_ID_COL].isin(vocab_item_set)]
    if len(part): item_feature_rows.append(part)
    del part, batch; gc.collect()

item_feat_df = pd.concat(item_feature_rows, ignore_index=True) if item_feature_rows else pd.DataFrame(columns=read_cols)
item_feat_df = item_feat_df.drop_duplicates(ITEM_ID_COL)
print('feature rows kept:', item_feat_df.shape)

def build_item_idx_mapper():
    keys = np.asarray(list(item_to_idx.keys()), dtype=np.int64)
    vals = np.asarray([item_to_idx[int(k)] for k in keys], dtype=np.int32)
    order = np.argsort(keys)
    keys = keys[order]; vals = vals[order]
    def mapper(arr):
        arr = np.asarray(arr, dtype=np.int64)
        pos = np.searchsorted(keys, arr)
        valid = pos < len(keys)
        if valid.any():
            chk = valid.copy(); chk[valid] = keys[pos[valid]] == arr[valid]; valid = chk
        out = np.zeros(len(arr), dtype=np.int32)
        if valid.any(): out[valid] = vals[pos[valid]]
        return out
    return mapper
map_vocab_items = build_item_idx_mapper()

feature_matrix = np.zeros((len(idx_to_item), len(feature_cols)), dtype=np.int32)
raw_feature_arrays = {c: np.zeros(len(idx_to_item), dtype=np.int32) for c in feature_cols}
cardinalities = []
if len(item_feat_df):
    rows = map_vocab_items(item_feat_df[ITEM_ID_COL].to_numpy(np.int64))
    for j, col in enumerate(feature_cols):
        s = item_feat_df[col]
        fill = -1 if pd.api.types.is_numeric_dtype(s) else '__NA__'
        codes, uniques = pd.factorize(s.fillna(fill), sort=False)
        enc = codes.astype(np.int32) + 1
        feature_matrix[rows, j] = enc
        raw_feature_arrays[col][rows] = enc
        cardinalities.append(int(len(uniques)))
else:
    cardinalities = [1 for _ in feature_cols]

# Valid item mask based on vertical_id when available.
valid_item_mask = np.ones(len(idx_to_item), dtype=bool); valid_item_mask[0] = False
if 'vertical_id' in feature_cols and len(item_feat_df):
    rows = map_vocab_items(item_feat_df[ITEM_ID_COL].to_numpy(np.int64))
    vertical = item_feat_df['vertical_id'].fillna(-1).astype('int16').to_numpy()
    known_valid = np.isin(vertical, list(VALID_VERTICALS))
    valid_item_mask[rows] = known_valid

print('feature_matrix:', feature_matrix.shape, 'MB:', round(feature_matrix.nbytes/1024**2, 1))
print('cardinalities:', dict(zip(feature_cols, cardinalities)))
del item_feature_rows, item_feat_df; gc.collect()


In [ ]:
# =========================
# 6. Feature-selection report and EDA
# =========================
feature_report = {
    'event_features': ['recent item_id sequence', 'recent eid sequence', 'position'],
    'item_features': feature_cols,
    'train_partitions': len(TRAIN_PATHS),
    'vocab_items': int(len(idx_to_item)-1),
    'training_examples': int(len(y)),
    'eval_users': int(len(eval_users)),
    'memory_strategy': 'bounded event slice + item_features filtered to vocab items',
}
with open(REPORT_DIR/'feature_selection.json', 'w', encoding='utf-8') as f: json.dump(feature_report, f, ensure_ascii=False, indent=2)
print(json.dumps(feature_report, ensure_ascii=False, indent=2))
if plt is not None:
    fig_dir = REPORT_DIR/'figures'; fig_dir.mkdir(parents=True, exist_ok=True)
    pd.Series(dict(eid_counts)).sort_values(ascending=False).head(30).sort_values().plot(kind='barh', figsize=(8,7), color='#4d6a8a')
    plt.title('Top event ids'); plt.tight_layout(); plt.savefig(fig_dir/'event_id_frequency.png', dpi=150); plt.close()
    pd.Series(dict(contact_item_counts)).sort_values(ascending=False).head(80).reset_index(drop=True).plot(figsize=(8,4), color='#b23a48')
    plt.title('Contact item popularity tail'); plt.tight_layout(); plt.savefig(fig_dir/'contact_item_tail.png', dpi=150); plt.close()


In [ ]:
# =========================
# 7. Dataset and one neural model
# =========================
class NextContactDataset(torch.utils.data.Dataset):
    def __init__(self, x_items, x_eids, y, n_items, negatives, seed):
        self.x_items=x_items; self.x_eids=x_eids; self.y=y; self.n_items=n_items; self.negatives=negatives; self.rng=np.random.default_rng(seed)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        pos=int(self.y[idx]); neg=self.rng.integers(1,self.n_items+1,size=self.negatives,dtype=np.int64); neg[neg==pos]=(pos%self.n_items)+1
        return {'seq_items':torch.from_numpy(self.x_items[idx].astype(np.int64,copy=False)), 'seq_eids':torch.from_numpy(self.x_eids[idx].astype(np.int64,copy=False)), 'mask':torch.from_numpy((self.x_items[idx]>0).astype(np.float32,copy=False)), 'pos_item':torch.tensor(pos,dtype=torch.long), 'neg_items':torch.from_numpy(neg)}

class FeatureSeqNeuralRecommender(nn.Module):
    def __init__(self, n_items, cardinalities, item_feature_matrix, max_eid, emb_dim=96, hidden_dim=160, max_seq_len=40):
        super().__init__()
        self.register_buffer('item_feature_matrix', torch.as_tensor(item_feature_matrix, dtype=torch.long))
        self.item_id_embedding = nn.Embedding(n_items+1, emb_dim, padding_idx=0)
        self.feature_embeddings = nn.ModuleList([nn.Embedding(max(1,int(c))+1, emb_dim, padding_idx=0) for c in cardinalities])
        self.eid_embedding = nn.Embedding(max_eid+1, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, emb_dim)
        self.item_proj = nn.Sequential(nn.LayerNorm(emb_dim), nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.LayerNorm(emb_dim))
        layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=4, dim_feedforward=hidden_dim*4, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.user_proj = nn.Sequential(nn.LayerNorm(emb_dim), nn.Linear(emb_dim, emb_dim), nn.Tanh())
        self.item_bias = nn.Embedding(n_items+1, 1)
        self.temperature = nn.Parameter(torch.tensor(0.07))
    def item_encode(self, item_idx):
        base = self.item_id_embedding(item_idx.clamp_min(0))
        flat = item_idx.reshape(-1).clamp_min(0)
        feats = self.item_feature_matrix[flat]
        feat_emb = 0
        for j, emb in enumerate(self.feature_embeddings): feat_emb = feat_emb + emb(feats[:,j])
        feat_emb = feat_emb.reshape(*item_idx.shape, -1)
        return self.item_proj(base + feat_emb)
    def user_encode(self, seq_items, seq_eids, mask):
        pos=torch.arange(seq_items.shape[1],device=seq_items.device).unsqueeze(0)
        x=self.item_encode(seq_items.clamp_min(0)) + self.eid_embedding(seq_eids.clamp(0,self.eid_embedding.num_embeddings-1)) + self.pos_embedding(pos)
        x=x*mask.unsqueeze(-1); key_padding=mask.eq(0); empty=mask.sum(1).eq(0)
        if empty.any(): key_padding=key_padding.clone(); key_padding[empty]=False
        enc=self.encoder(x, src_key_padding_mask=key_padding)
        pooled=(enc*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1.0)
        return self.user_proj(pooled)
    def score_items(self, user_vec, item_idx):
        item_vec=self.item_encode(item_idx); scores=(user_vec.unsqueeze(-2)*item_vec).sum(-1)/self.temperature.abs().clamp_min(0.02)
        return scores+self.item_bias(item_idx).squeeze(-1)
    def forward(self,batch):
        user_vec=self.user_encode(batch['seq_items'],batch['seq_eids'],batch['mask']); cand=torch.cat([batch['pos_item'].unsqueeze(1),batch['neg_items']],dim=1)
        return self.score_items(user_vec,cand)


In [ ]:
# =========================
# 8. Train
# =========================
n_items=len(idx_to_item)-1; max_eid=max(max(eid_counts.keys()) if eid_counts else 1, max(contact_eids) if contact_eids else 1)
perm=np.random.default_rng(SEED).permutation(len(y)); X_items=X_items[perm]; X_eids=X_eids[perm]; y=y[perm]
dataset=NextContactDataset(X_items,X_eids,y,n_items,NEGATIVES,SEED)
val_size=max(1,min(len(dataset)//20,20_000)); train_size=len(dataset)-val_size
train_ds,val_ds=random_split(dataset,[train_size,val_size],generator=torch.Generator().manual_seed(SEED))
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
model=FeatureSeqNeuralRecommender(n_items,cardinalities,feature_matrix,max_eid,EMB_DIM,HIDDEN_DIM,MAX_SEQ_LEN).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda'); best_val=float('inf')
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for batch in tqdm(train_loader,desc=f'epoch {epoch}/{EPOCHS}'):
        batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}; labels=torch.zeros(batch['pos_item'].shape[0],dtype=torch.long,device=DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type=='cuda'):
            loss=F.cross_entropy(model(batch),labels)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
    model.eval(); val_losses=[]; top1=[]
    with torch.no_grad():
        for batch in val_loader:
            batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}; labels=torch.zeros(batch['pos_item'].shape[0],dtype=torch.long,device=DEVICE); logits=model(batch)
            val_losses.append(float(F.cross_entropy(logits,labels).cpu())); top1.append(float((logits.argmax(1)==0).float().mean().cpu()))
    val_loss=float(np.mean(val_losses)); print(f'epoch={epoch} train_loss={np.mean(losses):.5f} val_loss={val_loss:.5f} val_top1={np.mean(top1):.5f}')
    if val_loss<best_val: best_val=val_loss; torch.save(model.state_dict(),MODEL_DIR/'model.pt')
model.load_state_dict(torch.load(MODEL_DIR/'model.pt',map_location=DEVICE)); model.eval(); print('best_val_loss:',best_val)


In [ ]:
# =========================
# 9. Candidate pool and submission
# =========================
pool=[]; seen=set()
for counter in [contact_item_counts,item_counts]:
    for idx,_ in counter.most_common(CANDIDATE_POOL_SIZE*3):
        if idx>0 and idx not in seen and idx < len(valid_item_mask) and valid_item_mask[idx]:
            pool.append(int(idx)); seen.add(int(idx))
            if len(pool)>=CANDIDATE_POOL_SIZE: break
    if len(pool)>=CANDIDATE_POOL_SIZE: break
if len(pool)<SUBMISSION_K: raise RuntimeError(f'candidate pool too small: {len(pool)}')
candidate_np=np.asarray(pool[:CANDIDATE_POOL_SIZE],dtype=np.int64); candidate_t=torch.tensor(candidate_np,dtype=torch.long,device=DEVICE)
print('candidate pool:',len(candidate_np))

def make_user_batch(users):
    seq_i=np.zeros((len(users),MAX_SEQ_LEN),dtype=np.int64); seq_e=np.zeros((len(users),MAX_SEQ_LEN),dtype=np.int64); masks=np.zeros((len(users),MAX_SEQ_LEN),dtype=np.float32); seen_sets=[]
    for n,u in enumerate(users):
        full=eval_histories.get(int(u),deque()); hist=list(full)[-MAX_SEQ_LEN:]; seen_sets.append({int(x[0]) for x in full})
        if hist:
            off=MAX_SEQ_LEN-len(hist); seq_i[n,off:]=[x[0] for x in hist]; seq_e[n,off:]=[x[1] for x in hist]; masks[n,off:]=1.0
    return seq_i,seq_e,masks,seen_sets

rows_out=[]
with torch.no_grad():
    cand_vec=model.item_encode(candidate_t); cand_bias=model.item_bias(candidate_t).squeeze(-1); temp=model.temperature.abs().clamp_min(0.02)
    for start in tqdm(range(0,len(eval_users),BATCH_SIZE),desc='predict users'):
        users=eval_users[start:start+BATCH_SIZE]; seq_i,seq_e,masks,seen_sets=make_user_batch(users)
        seq_i=torch.tensor(seq_i,dtype=torch.long,device=DEVICE); seq_e=torch.tensor(seq_e,dtype=torch.long,device=DEVICE); masks=torch.tensor(masks,dtype=torch.float32,device=DEVICE)
        user_vec=model.user_encode(seq_i,seq_e,masks); scores=(user_vec@cand_vec.T)/temp+cand_bias.unsqueeze(0)
        for i,s in enumerate(seen_sets):
            if s:
                hit=np.flatnonzero(np.isin(candidate_np,np.fromiter(s,dtype=np.int64)))
                if len(hit): scores[i,torch.tensor(hit,dtype=torch.long,device=DEVICE)] = -1e30
        _,top=torch.topk(scores,k=SUBMISSION_K,dim=1); rec_idx=candidate_np[top.cpu().numpy()]
        for user_id,recs in zip(users,rec_idx): rows_out.extend((int(user_id),int(idx_to_item[idx])) for idx in recs)
submission=pd.DataFrame(rows_out,columns=['user_id','item_id']).drop_duplicates(['user_id','item_id']); counts=submission.groupby('user_id').size()
print('submission shape:',submission.shape,'users:',submission['user_id'].nunique(),'min_k:',int(counts.min()),'max_k:',int(counts.max()))
assert submission['user_id'].nunique()==len(eval_users); assert counts.min()==SUBMISSION_K and counts.max()==SUBMISSION_K; assert submission.duplicated(['user_id','item_id']).sum()==0
submission.to_csv(SUBMISSION_FILE,index=False); print('saved:',SUBMISSION_FILE)


## If it is still too slow

Use the same notebook with smaller first-cell values:

```python
TRAIN_PARTITION_LIMIT = 8
MAX_VOCAB_ITEMS = 250_000
MAX_TRAIN_EXAMPLES = 350_000
CANDIDATE_POOL_SIZE = 4096
```

Quality mainly depends on using fresh train partitions and feature-aware candidates. The final ranking remains one neural network.
